# 04. 智能归因分析

职责3-智能归因：核心指标波动时，通过多维下钻和归因分析定位根因。

涵盖：单维贡献度、多维下钻、加性分解/Shapley、假设检验显著性验证。

## 1. 自动检测异常波动期

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from utils.data_loader import prepare_all
from attribution import detect_anomaly_period, dimension_contribution, multi_dim_drill_down

md = prepare_all()['multi_dim_sales'].copy()
md['period'] = md['month'].dt.strftime('%Y-%m')
periods = detect_anomaly_period(md, 'sales', 'period', threshold=0.2)
print('异常波动期:', periods)

## 2. 单维度贡献度

In [ ]:
base, current = periods
for dim in ['region', 'model', 'channel']:
    c = dimension_contribution(md, dim, 'sales', current, base)
    print(f'\n=== {dim} ===')
    print(c.head(3).to_string(index=False))

## 3. 多维逐层下钻

In [ ]:
drill = multi_dim_drill_down(md, ['region','channel','model'], 'sales', current, base, top_n=3)
display(drill)

## 4. 贡献度分解

In [ ]:
from attribution import additive_decomposition, waterfall_data
add = additive_decomposition(md, ['region','channel'], 'sales', current, base)
print('Top 5 贡献组合:')
print(add.head(5).to_string(index=False))

# 瀑布图数据
wf = waterfall_data(md, 'region', 'sales', current, base)
print('\n瀑布图数据:')
print(wf.to_string(index=False))

## 5. 假设检验 — 显著性验证

In [ ]:
from attribution import HypothesisTester
ht = HypothesisTester(alpha=0.05)

# 检验华南区域 当期 vs 基期 是否显著差异
a = md[(md['region']=='华南') & (md['period']==base)]['sales']
b = md[(md['region']=='华南') & (md['period']==current)]['sales']
print(ht.t_test(a, b))
print(ht.mann_whitney(a, b))

## 小结

- 自动检测波动期 + 多维下钻定位根因切片
- 贡献度分解量化各维度的贡献占比
- 假设检验确保归因结论具有统计显著性，避免过度解读